<a href="https://colab.research.google.com/github/fralfaro/MAT306/blob/main/docs/labs/lab_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# MAT306 - Laboratorio N°03





**Objetivo**: Aplicar técnicas avanzadas de manipulación y análisis de datos con pandas sobre un conjunto real de datos de contenido de Netflix, reforzando buenas prácticas y métodos eficientes sin recurrir a `groupby`, `merge`, `pivot`, ni `join`.



**Dataset**:

Trabajaremos con el archivo `netflix_titles.csv`, que contiene información sobre los títulos disponibles en la plataforma Netflix hasta el año 2021.

| Variable       | Clase     | Descripción                                                                 |
|----------------|-----------|------------------------------------------------------------------------------|
| show_id        | caracter  | Identificador único del título en el catálogo de Netflix.                   |
| type           | caracter  | Tipo de contenido: 'Movie' o 'TV Show'.                                     |
| title          | caracter  | Título del contenido.                                                       |
| director       | caracter  | Nombre del director (puede ser nulo).                                       |
| cast           | caracter  | Lista de actores principales (puede ser nulo).                              |
| country        | caracter  | País o países donde se produjo el contenido.                                |
| date_added     | fecha     | Fecha en la que el título fue agregado al catálogo de Netflix.              |
| release_year   | entero    | Año de lanzamiento original del título.                                     |
| rating         | caracter  | Clasificación por edad (por ejemplo: 'PG-13', 'TV-MA').                      |
| duration       | caracter  | Duración del contenido (minutos o número de temporadas para series).        |
| listed_in      | caracter  | Categorías o géneros en los que está clasificado el contenido.              |
| description    | caracter  | Breve sinopsis del contenido.                                               |




In [41]:
import pandas as pd

# Cargar datos
df = pd.read_csv('https://raw.githubusercontent.com/fralfaro/MAT306/main/docs/labs/data/netflix_titles.csv')
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...



### Parte 1: Limpieza y preparación

1. Revisar y describir el dataset:

   * ¿Cuántas filas y columnas tiene?
   * ¿Qué tipos de datos hay?
   * ¿Cuántos valores nulos hay por columna?

2. Transformar la columna `date_added` a tipo fecha.

3. Crear columnas auxiliares con `assign`:

   * Año (`year_added`)
   * Mes (`month_added`)



In [42]:
# Revisar y describir el dataset
print("Número de filas y columnas:", df.shape)
print("Tipos de datos:\n", df.dtypes)
print("Valores nulos por columna:\n", df.isnull().sum())

# Transformar la columna date_added a tipo fecha
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

# Crear columnas auxiliares con assign
df = df.assign(
    year_added=df['date_added'].dt.year,
    month_added=df['date_added'].dt.month
)

# Convertir month_added a tipo entero (Int64 para manejar NaNs)
df['month_added'] = df['month_added'].astype('Int64')


Número de filas y columnas: (8807, 12)

Tipos de datos:
 show_id         object
type            object
title           object
director        object
cast            object
country         object
date_added      object
release_year     int64
rating          object
duration        object
listed_in       object
description     object
dtype: object

Valores nulos por columna:
 show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64


## Parte 2: Técnicas avanzadas de pandas

4. Utilizar `.loc` para seleccionar películas (`type == 'Movie'`) que fueron agregadas después del año 2018.

5. Utilizar `str.contains()` y `str.extract()`:

   * Filtrar títulos que contienen la palabra 'love' (sin distinguir mayúsculas/minúsculas).
   * Extraer la duración en minutos para las películas desde la columna `duration`.

6. Aplicar `explode()` sobre la columna `listed_in` para obtener una fila por cada género.

7. Obtener un top 10 de géneros más frecuentes utilizando `value_counts()`.

8. Aplicar `where()` y `mask()` para marcar las películas de más de 120 minutos como contenido largo en una nueva columna.

9. Utilizar `.loc` para filtrar películas que cumplen con:

   * Más de 100 minutos de duración.
   * Rating igual a `'R'`.
   * País igual a `'United States'`.

10. Utilizar `.style` para formatear visualmente el top 10 de películas más largas.

In [44]:
#4.
movies_after_2018 = df.loc[(df['type'] == 'Movie') & (df['year_added'] > 2018)]
display("Películas agregadas después del año 2018:")
display(movies_after_2018.head())

#5.
love_titles = df.loc[df['title'].str.contains('love', case=False, na=False)]
display("Películas con 'love' en el título:")
display(love_titles.head())
movies_df = df.loc[df['type'] == 'Movie'].copy()
movies_df['duration_minutes'] = movies_df['duration'].str.extract(r'(\d+)').astype(float)
display("Películas con duración en minutos:")
display(movies_df.head())

#6.
df['listed_in_list'] = df['listed_in'].str.split(', ')
df_exploded = df.explode('listed_in_list')
display("DataFrame después de explode en listed_in:")
display(df_exploded.head())


#7.
top_10_genres = df_exploded['listed_in_list'].value_counts().head(10)
display("Top 10 géneros más frecuentes:")
display(top_10_genres)


#8.
movies_df_copy = movies_df.copy()
movies_df_copy['is_long_content'] = movies_df_copy['duration_minutes'].apply(lambda x: True if x > 120 else False)
display("Películas con columna 'is_long_content':")
display(movies_df_copy[['title', 'duration_minutes', 'is_long_content']].head())


#9.
filtered_movies = movies_df_copy.loc[
    (movies_df_copy['duration_minutes'] > 100) &
    (movies_df_copy['rating'] == 'R') &
    (movies_df_copy['country'] == 'United States')
]
display("Películas filtradas (duración > 100 min, rating 'R', país 'United States'):")
display(filtered_movies.head())


#10.
movies_df_sorted = movies_df_copy.dropna(subset=['duration_minutes']).sort_values(by='duration_minutes', ascending=False)
top_10_longest_movies = movies_df_sorted.head(10)
display("Top 10 películas más largas (formato con .style):")
display(top_10_longest_movies[['title', 'duration_minutes']].style.background_gradient(subset=['duration_minutes'], cmap='Blues'))

'\nPelículas agregadas después del año 2018:'

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,month_added,listed_in_list
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",2021.0,9,[Documentaries]
6,s7,Movie,My Little Pony: A New Generation,"Robert Cullen, José Luis Ucha","Vanessa Hudgens, Kimiko Glenn, James Marsden, ...",NaN,2021-09-24,2021,PG,91 min,Children & Family Movies,Equestria's divided. But a bright-eyed hero be...,2021.0,9,[Children & Family Movies]
7,s8,Movie,Sankofa,Haile Gerima,"Kofi Ghanaba, Oyafunmike Ogunlano, Alexandra D...","United States, Ghana, Burkina Faso, United Kin...",2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies","On a photo shoot in Ghana, an American model s...",2021.0,9,"[Dramas, Independent Movies, International Mov..."
9,s10,Movie,The Starling,Theodore Melfi,"Melissa McCarthy, Chris O'Dowd, Kevin Kline, T...",United States,2021-09-24,2021,PG-13,104 min,"Comedies, Dramas",A woman adjusting to life after a loss contend...,2021.0,9,"[Comedies, Dramas]"
12,s13,Movie,Je Suis Karl,Christian Schwochow,"Luna Wedler, Jannis Niewöhner, Milan Peschel, ...","Germany, Czech Republic",2021-09-23,2021,TV-MA,127 min,"Dramas, International Movies",After most of her family is murdered in a terr...,2021.0,9,"[Dramas, International Movies]"


"\nPelículas con 'love' en el título:"

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,month_added,listed_in_list
25,s26,TV Show,Love on the Spectrum,NaN,Brooke Satchwell,Australia,2021-09-21,2021,TV-14,2 Seasons,"Docuseries, International TV Shows, Reality TV",Finding love can be hard for anyone. For young...,2021.0,9,"[Docuseries, International TV Shows, Reality TV]"
158,s159,Movie,Love Don't Cost a Thing,Troy Byer,"Nick Cannon, Christina Milian, Kenan Thompson,...",United States,2021-09-01,2003,PG-13,101 min,"Comedies, Romantic Movies",A nerdy teen tries to make himself cool by ass...,2021.0,9,"[Comedies, Romantic Movies]"
159,s160,Movie,Love in a Puff,Pang Ho-cheung,"Miriam Chin Wah Yeung, Shawn Yue, Singh Hartih...",Hong Kong,2021-09-01,2010,TV-MA,103 min,"Comedies, Dramas, International Movies",When the Hong Kong government enacts a ban on ...,2021.0,9,"[Comedies, Dramas, International Movies]"
206,s207,Movie,"LSD: Love, Sex Aur Dhokha",Dibakar Banerjee,"Nushrat Bharucha, Anshuman Jha, Neha Chauhan, ...",India,2021-08-27,2010,TV-MA,112 min,"Dramas, Independent Movies, International Movies",This provocative drama examines how the voyeur...,2021.0,8,"[Dramas, Independent Movies, International Mov..."
227,s228,Movie,Really Love,Angel Kristi Williams,"Kofi Siriboe, Yootha Wong-Loi-Sing, Michael Ea...",United States,2021-08-25,2020,TV-MA,95 min,"Dramas, Independent Movies, Romantic Movies",A rising Black painter tries to break into a c...,2021.0,8,"[Dramas, Independent Movies, Romantic Movies]"


'\nPelículas con duración en minutos:'

...

,title,duration_minutes,is_long_content
0,Dick Johnson Is Dead,90.0,False
6,My Little Pony: A New Generation,91.0,False
7,Sankofa,125.0,True
9,The Starling,104.0,False
12,Je Suis Karl,127.0,True


"\nPelículas filtradas (duración > 100 min, rating 'R', país 'United States'):"

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,month_added,listed_in_list,duration_minutes,is_long_content
48,s49,Movie,Training Day,Antoine Fuqua,"Denzel Washington, Ethan Hawke, Scott Glenn, T...",United States,2021-09-16,2001,R,122 min,"Dramas, Thrillers",A rookie cop with one day to prove himself to ...,2021.0,9,"[Dramas, Thrillers]",122.0,True
81,s82,Movie,Kate,Cedric Nicolas-Troyan,"Mary Elizabeth Winstead, Jun Kunimura, Woody H...",United States,2021-09-10,2021,R,106 min,Action & Adventure,"Slipped a fatal poison on her final job, a rut...",2021.0,9,[Action & Adventure],106.0,False
131,s132,Movie,Blade Runner: The Final Cut,Ridley Scott,"Harrison Ford, Rutger Hauer, Sean Young, Edwar...",United States,2021-09-01,1982,R,117 min,"Action & Adventure, Classic Movies, Cult Movies","In a smog-choked dystopian Los Angeles, blade ...",2021.0,9,"[Action & Adventure, Classic Movies, Cult Movies]",117.0,False
139,s140,Movie,Do the Right Thing,Spike Lee,"Danny Aiello, Ossie Davis, Ruby Dee, Richard E...",United States,2021-09-01,1989,R,120 min,"Classic Movies, Comedies, Dramas","On a sweltering day in Brooklyn, simmering rac...",2021.0,9,"[Classic Movies, Comedies, Dramas]",120.0,False
144,s145,Movie,House Party,Reginald Hudlin,"Christopher Reid, Christopher Martin, Robin Ha...",United States,2021-09-01,1990,R,104 min,"Comedies, Cult Movies","Grounded by his strict father, Kid risks life ...",2021.0,9,"[Comedies, Cult Movies]",104.0,False


'\nTop 10 películas más largas (formato con .style):'

,title,duration_minutes
4253,Black Mirror: Bandersnatch,312.000000
717,Headspace: Unwind Your Mind,273.000000
2491,The School of Mischief,253.000000
2487,No Longer kids,237.000000
2484,Lock Your Girls In,233.000000
2488,Raya and Sakina,230.000000
166,Once Upon a Time in America,229.000000
7932,Sangam,228.000000
1019,Lagaan,224.000000
4573,Jodhaa Akbar,214.000000




### Pregunta Desafío

11. ¿Cuáles son las combinaciones más frecuentes de género y rating en el dataset?
    (Sugerencia: utilizar `value_counts` con `subset=["genre", "rating"]` después de aplicar `explode()`).



### Bonus: Análisis de duplicados y limpieza

12. ¿Existen películas con el mismo nombre (`title`) pero con distinto año de lanzamiento (`release_year`)?
13. ¿Cuántos títulos únicos hay en total en la columna `title`?





In [54]:
# 11.
df_exploded_cleaned = df_exploded.dropna(subset=['listed_in_list', 'rating'])
genre_rating_combinations = df_exploded_cleaned.value_counts(subset=['listed_in_list', 'rating']).head(10)

display("Combinaciones más frecuentes de género y rating:")
display(genre_rating_combinations)


# 12.
duplicate_titles = df.groupby('title')['release_year'].nunique()
titles_with_different_years = duplicate_titles[duplicate_titles > 1]

display("Títulos de películas con el mismo nombre pero distinto año de lanzamiento:")
if titles_with_different_years.empty:
    display("No se encontraron películas con el mismo nombre pero distinto año de lanzamiento.")
else:
    display(titles_with_different_years)


# 13.
unique_titles_count = df['title'].nunique()

display("Número total de títulos únicos:")
display(unique_titles_count)

'Combinaciones más frecuentes de género y rating:'

listed_in_list          rating
International Movies    TV-MA     1130
                        TV-14     1065
Dramas                  TV-MA      830
International TV Shows  TV-MA      714
Dramas                  TV-14      693
International TV Shows  TV-14      472
Comedies                TV-14      465
TV Dramas               TV-MA      434
Comedies                TV-MA      431
Dramas                  R          375
Name: count, dtype: int64

'Títulos de películas con el mismo nombre pero distinto año de lanzamiento:'

'No se encontraron películas con el mismo nombre pero distinto año de lanzamiento.'

'\nNúmero total de títulos únicos:'

8807